In [ ]:
import pandas as pd
import wbdata

In [ ]:
from _helpers_notebooks import mock_snakemake
snakemake = mock_snakemake(
            "prepare_wacc",
    )

### File paths

In [ ]:
wacc_fn = snakemake.input.wacc

# Output
wacc_clustered_fn = snakemake.output.wacc
wacc_latex_fn = snakemake.output.wacc_latex

# config
config = snakemake.config

### Get WACC data

In [ ]:
# Read the CSV data
wacc = pd.read_csv(wacc_fn, comment='#', index_col="iso_alpha")


### Cluster per region


Map each country in the production data to its region using the region definitions from the config file.


In [ ]:
# Load region definitions from config file
regions = config['regions']

In [ ]:
# Build a mapping from World Bank's own country name to ISO-3 code.
# Using World Bank's own metadata (rather than pycountry) avoids name-matching
# mismatches for countries whose WB display name differs from pycountry's
# (e.g. WB "Turkiye" vs pycountry "Türkiye", WB "Korea, Rep." vs pycountry
# "Korea, Republic of"), which previously caused those countries' GDP rows to be
# dropped and, for single-country regions, produced NaN GDP-weighted WACC.
country_name_to_iso = {c["name"]: c["id"] for c in wbdata.get_countries()}

# Build a mapping from ISO-3 code to region (config regions now use ISO-3 codes directly)
iso_to_region = {}
for region, iso_codes in regions.items():
    for code in iso_codes:
        iso_to_region[code] = region

In [ ]:
# Map each row in production to its region
wacc['region'] = wacc.index.map(lambda iso: iso_to_region.get(iso, 'Other'))

### 

### Cluster: Weighted by GDP

In [ ]:
# --- fetch GDP (current USD) from World Bank ---
gdp_raw = wbdata.get_dataframe(
    {"NY.GDP.MKTP.CD": "gdp_usd"},
    date="2024",
)
gdp = gdp_raw.dropna(subset=["gdp_usd"]).reset_index()
gdp = gdp.rename(columns={"country": "country_name"})

# map country name → ISO-3 so we can merge on the wacc index
gdp["country_code"] = gdp["country_name"].map(
    lambda name: country_name_to_iso.get(name)
)
gdp = gdp.dropna(subset=["country_code"])

# --- attach GDP weights ---
wacc_gdp = wacc.merge(gdp[["country_code", "gdp_usd"]], left_index=True, right_on="country_code", how="left")

# fill missing GDP with regional median so no country is dropped
wacc_gdp["gdp_usd"] = wacc_gdp.groupby("region")["gdp_usd"].transform(
    lambda x: x.fillna(x.median())
)

def weighted_mean(df, value_col, weight_col):
    return (df[value_col] * df[weight_col]).sum() / df[weight_col].sum()

wacc_by_region = (
    wacc_gdp.groupby("region")
    .apply(lambda g: pd.Series({
        "wacc":                  weighted_mean(g, "wacc", "gdp_usd"),
        "wacc_real":             weighted_mean(g, "wacc_real", "gdp_usd"),
        "risk_free_rate":        weighted_mean(g, "risk_free_rate", "gdp_usd"),
        "country_risk_premium":  weighted_mean(g, "country_risk_premium", "gdp_usd"),
        "equity_risk_premium":   weighted_mean(g, "equity_risk_premium", "gdp_usd"),
        "cost_of_equity":        weighted_mean(g, "cost_of_equity", "gdp_usd"),
        "cost_of_debt":          weighted_mean(g, "cost_of_debt", "gdp_usd"),
        "n_countries":           len(g),
        "wacc_spread":           g["wacc"].max() - g["wacc"].min(),  # diagnostics
    }))
    .reset_index()
)

In [ ]:
wacc_by_region.set_index("region", inplace=True)

### Save clustered WACC

In [ ]:
wacc_by_region

In [ ]:
# Filter for relevant columns
wacc_by_region = wacc_by_region[["wacc_real", "n_countries", "wacc_spread"]]

In [ ]:
# Calculate global GDP-weighted average and append as last row
global_avg = pd.Series({
    "wacc_real":    weighted_mean(wacc_gdp, "wacc_real", "gdp_usd"),
    "n_countries":  len(wacc_gdp),
    "wacc_spread":  wacc_gdp["wacc"].max() - wacc_gdp["wacc"].min(),
}, name="global_weighted_avg")

wacc_by_region = pd.concat([wacc_by_region, global_avg.to_frame().T])

In [ ]:
wacc_by_region.index.name = "region"

In [ ]:
wacc_by_region.to_csv(wacc_clustered_fn)

### Save latex table

In [ ]:
wacc_table = (
    wacc_by_region[["wacc_real"]]
    .rename(columns={"wacc_real": "Weighted average cost of capital in \%"})
    .rename(index=lambda region: config["region_nice_names"].get(region, region)).mul(100)
    .round(1)
)

# wacc_table *= 100  # convert to percentage
wacc_table = wacc_table.rename(index={"global_weighted_avg": "Global weighted average"})

# Flatten index so Region and value header are on the same header row in LaTeX.
wacc_table_out = wacc_table.rename_axis("Region").reset_index()


def escape_latex(text):
    return str(text).replace("&", r"\&")


value_col = wacc_table_out.columns[1]
column_name = escape_latex(value_col)
newline = r"\\"
latex_lines = [
    r"\begin{table}[htbp]",
    r"\centering",
    r"\caption{Regional weighted average cost of capital (real), in \%.}",
    r"\begin{tabular}{lr}",
    r"\toprule",
    f"\\textbf{{Region}} & \\textbf{{{column_name}}} {newline}",
    r"\midrule",
]

for _, row in wacc_table_out.iterrows():
    region = escape_latex(row["Region"])
    value = row[value_col]
    latex_lines.append(f"{region} & {value:.1f} {newline}")

latex_lines.extend([
    r"\bottomrule",
    r"\end{tabular}",
    r"\end{table}",
])

with open(wacc_latex_fn, "w", encoding="utf-8") as f:
    f.write("\n".join(latex_lines) + "\n")

wacc_table_out

### Further analysis

In [ ]:
wacc_by_region.loc[:,["wacc_real", "wacc_spread"]].round(3)

In [ ]:
wacc.loc[wacc.region.isin(["Middle_East","North_West_Africa"])].loc[:,["country_name", "wacc_real", "wacc", "region"]].sort_values("region").round(3)